# Data ingest and transformation

# FIX

## - Cap modified column titles
## - scaling by scenario probabilities
## - charts

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


In [ ]:
# Load the provided JSON file
file_path = 'DE_L_results_T_10_delta_5_scen_21_trial_1_inv_2_cap._1_cap.inc._5.json'
with open(file_path, 'r') as file:
    data = json.load(file)
    
# Create the combined data dictionary according to the provided mapping
combined_data = {
    "F_tender_schedule": data.get("F", {}),
    "Y_manufacturers_producing_period": data.get("Y", {}),
    "W_manufacturers_participate_tender": data.get("W", {}),
    "L_capacity_extension_decision": data.get("L", {}),
    "Q_commitment_amounts": data.get("Q", {}),
    "X_production_amounts": data.get("X", {}),
    "I_inventory_level": data.get("I", {}),
    "Vc_vaccinated_children": data.get("Vc", {}),
    "S_unvaccinated_children": data.get("S", {})
}

unique_producers = set(combined_data['Y_manufacturers_producing_period'].keys())
unique_vaccines = set(combined_data['Q_commitment_amounts'].keys())

# unique_producers, unique_vaccines

In [ ]:
# Adjusting the ordering function to handle nested dictionaries with non-numeric keys
def order_data(data_dict):
    ordered_data = {}
    for key, value in data_dict.items():
        if isinstance(value, dict):
            ordered_data[key] = {k: order_data(v) if isinstance(v, dict) else v for k, v in sorted(value.items(), key=lambda item: int(item[0]) if item[0].isdigit() else item[0])}
        else:
            ordered_data[key] = value
    return ordered_data

# Apply ordering to combined_data
L_ordered = order_data(combined_data['L_capacity_extension_decision'])

# Convert the 'L_capacity_extension_decision' data into a DataFrame for better visualization
L_df = pd.DataFrame(L_ordered).sort_index()
# Rotate the DataFrame
L_df_rotated = L_df.transpose()
# Order the columns from 1 to 10
ordered_columns = [str(i) for i in range(1, 11)]
L_df_rotated_ordered = L_df_rotated[ordered_columns]
# Reset the index to make manufacturers the first column
L_df_rotated_ordered.reset_index(inplace=True)
L_df_rotated_ordered.rename(columns={'index': 'Manufacturer'}, inplace=True)

L_df_rotated_ordered = L_df_rotated_ordered.sort_values(by='Manufacturer').reset_index(drop=True)
# L_df_rotated_ordered


In [ ]:
def transform_row(row):
    row = row.copy()
    if row.iloc[1] == 5.0:
        row.iloc[1] = 1.5
    else:
        row.iloc[1] = 1.0

    for i in range(2, len(row)):
        if row.iloc[i] == 5.0:
            row.iloc[i] = row.iloc[i-1] + 0.5
        else:
            row.iloc[i] = row.iloc[i-1]
    
    return row

# Apply the transformation to each row
transformed_capacity_increase = L_df_rotated_ordered.apply(transform_row, axis=1)
# Rename all columns except the first one to integers
transformed_capacity_increase.rename(columns={col: int(col) for col in transformed_capacity_increase.columns[1:]}, inplace=True)
transformed_capacity_increase.set_index(transformed_capacity_increase.columns[0], inplace=False)

In [ ]:
# Load the uploaded Excel file
file_path = '../../data/production_capacity_scenarios/production_capacity_scenarios.xlsx'
xls = pd.ExcelFile(file_path)

# Read the "master capacity" sheet
df_master_capacity = pd.read_excel(xls, sheet_name='master_capacity')
# df_master_capacity = df_master_capacity.sort_values(by='Manufacturer')
df_master_capacity.set_index(df_master_capacity.columns[0], inplace=False)

# df_master_capacity

In [ ]:

# Performing the multiplication
adjusted_capacity = df_master_capacity.iloc[:, 1:11] * (transformed_capacity_increase.iloc[:, 1:11]/1)

# Adding the Manufacturer column back to the selected dataframe
adjusted_capacity['Manufacturer'] = df_master_capacity['Manufacturer']

adjusted_capacity = adjusted_capacity[[adjusted_capacity.columns[-1]] + list(adjusted_capacity.columns[:-1])]

adjusted_capacity.set_index(adjusted_capacity.columns[0], inplace=False)
# adjusted_capacity

## Transform cap increase and capacity to one DF

# Graphing

In [ ]:
import matplotlib.pyplot as plt

def plot_capacity_over_time(capacity_data, producers):
    """
    Plots the capacity over time for the given list of producers.

    Parameters:
    capacity_data (pd.DataFrame): DataFrame containing the capacity data with 'Manufacturer' column.
    producers (list): List of producers to plot.

    Returns:
    None
    """
    # Filter the capacity data for the specified producers
    filtered_data = capacity_data[capacity_data['Manufacturer'].isin(producers)]
    
    # Set the 'Manufacturer' as the index for easy plotting
    filtered_data.set_index('Manufacturer', inplace=True)
    
    # Plot each producer's capacity over time
    for producer in producers:
        if producer in filtered_data.index:
            plt.figure(figsize=(10, 6))
            plt.plot(filtered_data.columns, filtered_data.loc[producer], marker='o', linestyle='-')
            plt.title(f'Capacity Over Time for {producer}')
            plt.xlabel('Time Period')
            plt.ylabel('Capacity')
            plt.grid(True)
            plt.show()

# Example usage of the function
example_producers = ['Serum_Institute']
plot_capacity_over_time(adjusted_capacity, example_producers)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Subset capacity data based on the producer list
# subset_capacity_data = final_df[final_df['Manufacturer'].isin(unique_producers_list)]

# Define the function to plot production with capacity extension
def plot_production_with_capacity_extension(producers, combined_data, vaccines_to_filter):
    # Define a color map for vaccines
    vaccine_colors = {
        "DT": "darkolivegreen",
        "DTwP": "green",
        "DTwP-Hib": "greenyellow",
        "Hexa": "red",
        "Penta": "orange",
        "HepB": "yellow",
        "IPV": "lemonchiffon",
        "Hib": "lightgoldenrodyellow",
        "HepB": "lime",
        "OPV": "bisque",
        "Penta": "antiquewhite",
        "Rota": "aqua",
        "TT": "coral",
        "HPV": "gold",
        'PCV': "tomato",
        'Td': "burlywood",
        'Measles': "blue",
        'MMR': "aliceblue",
        'MR': "cadetblue",
    }

    for producer in producers:
        # Initialize dictionary to store production amounts by vaccine and time period for the specified producer
        producer_production = {}
        producer_capacity = {}

        # Collect data for the specified producer for the specified vaccines in productions
        for vaccine, producers_data in combined_data["X_production_amounts"].items():
            if vaccine in vaccines_to_filter and producer in producers_data:
                for time_period, production in producers_data[producer].items():
                    if time_period not in producer_production:
                        producer_production[time_period] = {}
                    if vaccine not in producer_production[time_period]:
                        producer_production[time_period][vaccine] = 0
                    # Sum the values if production is a nested dictionary
                    if isinstance(production, dict):
                        producer_production[time_period][vaccine] += sum(production.values())
                    else:
                        producer_production[time_period][vaccine] += production

        # Prepare data for visualization
        time_periods = sorted(producer_production.keys(), key=int)
        vaccines = sorted(set(vaccine for tp_data in producer_production.values() for vaccine in tp_data))

        production_data = {vaccine: [producer_production.get(tp, {}).get(vaccine, 0) for tp in time_periods] for vaccine in vaccines}

        # Convert to DataFrame for easier manipulation
        df_production = pd.DataFrame(production_data, index=time_periods)

        # Collect capacity extension decisions for the specified producer
        capacity_extension = combined_data["L_capacity_extension_decision"].get(producer, {})
        capacity_extension_data = [capacity_extension.get(tp, 0) for tp in time_periods]

        # Plotting the data
        bar_width = 0.5
        index = np.arange(len(time_periods))

        fig, ax1 = plt.subplots(figsize=(14, 8))

        # Stacked bar for productions
        bottom_production = np.zeros(len(time_periods))
        for vaccine in vaccines:
            color = vaccine_colors.get(vaccine, "gray")  # Default to gray if vaccine not in color map
            ax1.bar(index, df_production[vaccine], bar_width, label=f'{vaccine} Production', bottom=bottom_production, color=color)
            bottom_production += df_production[vaccine]

        # Adding upward facing arrows for capacity extension
        for i, (tp, capacity) in enumerate(zip(time_periods, capacity_extension_data)):
            if capacity > 0:
                ax1.annotate('↑', (index[i], bottom_production[i] / 2), ha='center', va='center', fontsize=15, color='black')

        # # Plotting the capacity data as a black line
        # ax2 = ax1.twinx()
        # ax2.plot(index, capacity_values, label='Capacity', color='black', marker='o')
        # ax2.set_ylabel('Capacity')

        # Adding labels and titles
        ax1.set_xlabel('Time Period')
        ax1.set_ylabel('Production Amount')
        ax1.set_title(f'{producer} Production and Capacity Extension Over Time by Vaccine')
        ax1.set_xticks(index)
        ax1.set_xticklabels(time_periods)

        # Adding legend
        handles1, labels1 = ax1.get_legend_handles_labels()
        # handles2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(handles1 , labels1 , loc='upper left')

        plt.grid(True)
        plt.tight_layout()
        plt.show()

# Example call to the function with subsetted capacity data
plot_production_with_capacity_extension(unique_producers, combined_data, unique_vaccines)
